# DM-SR — COLMAP geometry baseline (Kaggle)

Execute **somente depois** de encerrar uma sessão do treino principal DM-NeRF, para não disputar a GPU/cota. Ative GPU + Internet e use **Save Version → Save & Run All**.

Este baseline usa as poses e intrínsecos fornecidos pelo DM-SR e reconstrói apenas geometria multi-view; não é baseline de decomposição semântica/instâncias.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

def run(*args): subprocess.run([str(x) for x in args],check=True)
PIN='8d0f74859103f13c57b3a00190cfcbc682470d7c'
BASE=Path('/kaggle/temp/dmsr-colmap-baseline')
OUT=Path('/kaggle/working/dmsr-colmap-baseline')
if BASE.exists(): shutil.rmtree(BASE)
if OUT.exists(): shutil.rmtree(OUT)
BASE.mkdir(parents=True)
REPO=BASE/'sdf-generator'; WORK=BASE/'work'
run('git','clone','https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'checkout','--detach',PIN)
run('apt-get','update','-qq')
run('apt-get','install','-y','-qq','colmap')
run(sys.executable,'-m','pip','install','-q','h5py','pillow','trimesh','scipy')
run(sys.executable,REPO/'tools/dmnerf/prepare.py','--work',WORK)
SCENE=WORK/'DM-NeRF/data/dmsr/study'
print('COLMAP:',subprocess.check_output(['colmap','-h'],text=True).splitlines()[0])
print('SCENE:',SCENE)


In [ ]:
run(sys.executable,REPO/'tools/baselines/run_colmap_dmsr.py',SCENE,OUT/'run','--source-count','10','--near','4','--far','15')


In [ ]:
run(sys.executable,REPO/'tools/evaluate_mesh_distance.py',OUT/'run/dense/fused.ply',SCENE/'study.ply','--samples','200000','--thresholds','0.01','0.02','0.05','--out',OUT/'geometry_metrics.json')
archive=shutil.make_archive('/kaggle/working/dmsr-colmap-baseline-bundle','zip',OUT)
print('RESULTS:',archive)


Envie `dmsr-colmap-baseline-bundle.zip` para revisão. O resultado só entra na dissertação se a pose/convenção e a reconstrução forem validadas e o baseline completar sem ajustes ad hoc.
